In [ ]:
# ============================================================================================
# CELL 1 OF 4 - dim_carc AIDP WORKFLOW FRAMEWORK
# Config resolution, optional local debug, audit-derived window, open batch, active run,
# source/target path resolution, and target state validation.
#
# --------------------------------------------------------------------------------------------
# CHANGE HISTORY:
# --------------------------------------------------------------------------------------------
# Modification log for this notebook, newest first: date | JIRA | change.
# Changed lines are tagged "<JIRA> (<date>)" in the code; search the tag to find them.
# --------------------------------------------------------------------------------------------
#
# 2026-11-26 | n/a | SCD2 write cell: spark.sql.constraintPropagation.enabled set to false before
#   the SCD2 plan to avoid the Spark 3.5 "key not found: client_id#..." error raised by
#   localCheckpoint() on the union of changed and soft-deleted keys. No change to results.
# 2026-11-25 | n/a | Release 2:
#   - Source: the CARC RARC Seed Mapping workbook (.xlsx) instead of the R1 CSV, read from the
#     same Object Storage location (common.ref_data.bucket_name / ref_data_src_files) and
#     parsed with openpyxl. Sheet: tables.dim_carc.source_sheet_name (default "CARC Seed").
#   - Cell 2 now holds the R2 expansion and rules that were applied once by
#     qa/dim_carc_r2_seed_reload.ipynb (group expansion of '/'-separated and "Per Group &
#     Exclusion Lists" treatments, exclusion list, PR handling, Denial? / Actionable?,
#     category renames, deadline_check_ind), so a new workbook only needs a rerun.
#   - R2 table structure: client_id, default_carc_sub_cause, carc_notes,
#     reasoning_actionability_basis, actionable_ind, deadline_check_ind,
#     record_updated_datetime. Business key client_id + carc_code (config business_key
#     "client_id,carc_code"; "carc_code" is still accepted).
#   - record_source_system_name is optional: when it is not configured it is derived from the
#     workbook Read Me sheet (e.g. CARC_RARC_SEED_MAPPING_V0.3_2026-10-26).
#   - client_id: workflow parameter client_id, else tables.dim_carc.client_id, else 1.
#   - Local debug: local.debug.source_path overrides the source workbook path.
#   - Cell 3: SCD2 keyed on client_id + carc_code; expired rows also set
#     record_updated_datetime; schema compared by column name.
#   - Cell 4: the client's active rows must equal the prepared rows after the write.
# ============================================================================================
import json
import traceback
from datetime import datetime, timezone

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
DEFAULT_WINDOW_START_TS = datetime.strptime("2026-06-16 00:00:00.000", "%Y-%m-%d %H:%M:%S.%f")
FUTURE_EXPIRY_VALUE = "2099-12-31 23:59:59"
DEFAULT_SURROGATE_KEY = -1
DEFAULT_ETL_RUN_ID = "c0000000-0000-0000-0000-000000000000"
DEFAULT_CARC_CODE = "UNKNOWN"


def _conf_bool(key, default=False):
    try:
        return str(spark.conf.get(key, str(default).lower())).strip().lower() in ("true", "1", "yes", "y")
    except Exception:
        return bool(default)


LOCAL_DEBUG = _conf_bool("local.debug.enabled", False)
ENABLE_DEBUG_WRITE = LOCAL_DEBUG and _conf_bool("local.debug.write.enabled", False)
ENABLE_TARGET_WRITE = _conf_bool("target.write.enabled", True) if not LOCAL_DEBUG else ENABLE_DEBUG_WRITE

LOCAL_DEBUG_CONFIG = {
    "config_param": spark.conf.get("local.debug.config_param", None),
    "flow_type": spark.conf.get("local.debug.flow_type", "REFERENCE_DIMENSIONS"),
    "PARAM1": spark.conf.get("local.debug.PARAM1", None),
    "PARAM2": spark.conf.get("local.debug.PARAM2", None),
    "limit_rows": int(spark.conf.get("local.debug.limit_rows", "1000")),
    "active_etl_batch_id": spark.conf.get("local.debug.active_etl_batch_id", "LOCAL_DEBUG_BATCH_001"),
    "active_etl_run_id": spark.conf.get("local.debug.active_etl_run_id", "LOCAL_DEBUG_RUN_001"),
    "batch_load_layer_nm": spark.conf.get("local.debug.batch_load_layer_nm", "REFERENCE"),
    "batch_pipeline_stage": spark.conf.get("local.debug.batch_pipeline_stage", "CSV_TO_REFERENCE"),
}


def _get_param(key, default=None):
    if LOCAL_DEBUG:
        value = LOCAL_DEBUG_CONFIG.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    try:
        value = oidlUtils.parameters.getParameter(key, None)  # type: ignore[name-defined]
        if value is not None and str(value).strip():
            return str(value).strip()
    except Exception:
        pass
    try:
        value = spark.conf.get(key, None)
        if value is not None and str(value).strip():
            return str(value).strip()
    except Exception:
        pass
    try:
        value = dbutils.widgets.get(key)  # type: ignore[name-defined]
        if value is not None and str(value).strip():
            return str(value).strip()
    except Exception:
        pass
    return default


def _require(value, label):
    if value is None or not str(value).strip():
        raise ValueError(f"{label} is required")
    return str(value).strip()


def load_json_config(path):
    return json.loads("\n".join(row.value for row in spark.read.text(_require(path, "config_param")).collect()))


def get_flow_config(config, flow_type):
    transactions = config.get("transactions", {})
    normalized = {str(k).strip().upper(): v for k, v in transactions.items()}
    key = _require(flow_type, "flow_type").upper()
    if key not in normalized:
        raise ValueError(f"flow_type '{key}' not found. Available values: {sorted(normalized)}")
    return normalized[key]


def table_fqn(catalog, schema, table):
    return f"{_require(catalog, 'catalog')}.{_require(schema, 'schema')}.{_require(table, 'table')}"


def make_oci_uri(bucket, namespace, path):
    return f"oci://{_require(bucket, 'bucket')}@{_require(namespace, 'namespace')}/{str(path).strip('/')}"


def storage_exists(path):
    try:
        jvm = spark.sparkContext._jvm
        uri = jvm.java.net.URI(str(path))
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(uri, spark.sparkContext._jsc.hadoopConfiguration())
        return bool(fs.exists(jvm.org.apache.hadoop.fs.Path(str(path))))
    except Exception:
        return False


def get_table_location(table_name):
    try:
        return str(spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]["location"]).rstrip("/")
    except Exception:
        return None


def log(level, message, **extra):
    print(json.dumps({
        "ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"),
        "level": str(level).upper(),
        "message": str(message),
        "extra": extra,
    }, default=str))


def parse_param_timestamp(value, name, end_of_day=False):
    raw = _require(value, name)
    if raw.lower() == "current_timestamp()":
        return datetime.now(timezone.utc).replace(tzinfo=None)
    for fmt in (
        "%Y-%m-%d %H:%M:%S.%fZ", "%Y-%m-%dT%H:%M:%S.%fZ",
        "%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d",
    ):
        try:
            dt = datetime.strptime(raw, fmt)
            if fmt == "%Y-%m-%d" and end_of_day:
                dt = dt.replace(hour=23, minute=59, second=59, microsecond=999000)
            return dt
        except ValueError:
            pass
    parsed = datetime.fromisoformat(raw.replace("Z", "+00:00"))
    return parsed.astimezone(timezone.utc).replace(tzinfo=None) if parsed.tzinfo else parsed


def truncate_to_millis(value):
    return value.replace(microsecond=(value.microsecond // 1000) * 1000)


def format_utc_millis_z(value):
    return truncate_to_millis(value).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + "Z"


def sql_escape(value):
    return str(value if value is not None else "").replace("'", "''")


def _trim_upper_col(name):
    return F.upper(F.trim(F.col(name).cast("string")))


def _trim_upper_lit(value):
    return F.upper(F.trim(F.lit(str(value))))


def get_audit_derived_window(audit_table, flow_type_value, target_name, load_layer, stage):
    scoped = (spark.table(audit_table)
        .filter(_trim_upper_col("flow_type") == _trim_upper_lit(flow_type_value))
        .filter(_trim_upper_col("tgt_table_name") == _trim_upper_lit(target_name))
        .filter(_trim_upper_col("load_layer_nm") == _trim_upper_lit(load_layer))
        .filter(_trim_upper_col("pipeline_stage") == _trim_upper_lit(stage)))
    success = (scoped
        .filter(_trim_upper_col("etl_run_status") == _trim_upper_lit("SUCCESS"))
        .orderBy(F.col("etl_load_ts").desc(), F.col("etl_run_start_ts").cast("timestamp").desc())
        .select(F.col("etl_run_start_ts").cast("timestamp").alias("ts"))
        .limit(1).collect())
    start = success[0]["ts"] if success and success[0]["ts"] else DEFAULT_WINDOW_START_TS
    running = (scoped
        .filter(_trim_upper_col("etl_run_status") == _trim_upper_lit("RUNNING"))
        .filter(F.col("etl_run_end_ts").isNull())
        .filter(F.col("etl_run_start_ts").cast("timestamp") > F.lit(start))
        .orderBy(F.col("etl_load_ts").asc(), F.col("etl_run_start_ts").cast("timestamp").asc())
        .select(F.col("etl_run_start_ts").cast("timestamp").alias("ts"))
        .limit(1).collect())
    end = running[0]["ts"] if running and running[0]["ts"] else datetime.now(timezone.utc).replace(tzinfo=None)
    return {"param1_value": format_utc_millis_z(start), "param2_value": format_utc_millis_z(end)}


def get_open_batch_row(batch_table, source_system, flow, run, batch_layer, batch_stage):
    rows = spark.sql(f"""
        SELECT etl_batch_id FROM {batch_table}
        WHERE UPPER(TRIM(source_system_name)) = UPPER(TRIM('{sql_escape(source_system)}'))
          AND UPPER(TRIM(flow_type)) = UPPER(TRIM('{sql_escape(flow)}'))
          AND UPPER(TRIM(run_type)) = UPPER(TRIM('{sql_escape(run)}'))
          AND UPPER(TRIM(load_layer_nm)) = UPPER(TRIM('{sql_escape(batch_layer)}'))
          AND UPPER(TRIM(pipeline_stage)) = UPPER(TRIM('{sql_escape(batch_stage)}'))
          AND UPPER(TRIM(batch_status)) != 'CLOSED'
        ORDER BY batch_start_ts DESC LIMIT 1
    """).collect()
    if not rows or not rows[0]["etl_batch_id"]:
        raise ValueError("No latest open batch row found for the configured scope")
    return str(rows[0]["etl_batch_id"])


def get_active_etl_run_id(audit_table, etl_batch_id, target_table):
    rows = spark.sql(f"""
        SELECT etl_run_id FROM {audit_table}
        WHERE etl_batch_id = '{sql_escape(etl_batch_id)}'
          AND UPPER(TRIM(tgt_table_name)) = UPPER(TRIM('{sql_escape(target_table)}'))
          AND UPPER(TRIM(etl_run_status)) = 'RUNNING'
          AND etl_run_end_ts IS NULL
        ORDER BY etl_run_start_ts DESC LIMIT 1
    """).collect()
    if not rows or not rows[0]["etl_run_id"]:
        raise ValueError(f"No RUNNING audit row found for etl_batch_id={etl_batch_id}, table={target_table}")
    return str(rows[0]["etl_run_id"])


config_param = _require(_get_param("config_param"), "config_param")
flow_type = _require(_get_param("flow_type", "REFERENCE_DIMENSIONS"), "flow_type").upper()
source_system_name = str(_get_param("source_system_name", f"{flow_type}_FTP")).strip()
run_type = str(_get_param("run_type", "HIST")).strip().upper()
workflow_nm = str(_get_param("audit_job_name", _get_param("job_name", "AIDP_DIM_CARC"))).strip()
load_layer_nm = str(_get_param("audit_load_layer_nm", _get_param("load_layer_nm", "REFERENCE"))).strip().upper()
pipeline_stage = str(_get_param("audit_pipeline_stage", _get_param("pipeline_stage", "CSV_TO_REFERENCE"))).strip().upper()

config = load_json_config(config_param)
txn_cfg = get_flow_config(config, flow_type)
common_cfg = config.get("common", {})
runtime_cfg = common_cfg.get("runtime", {})
audit_cfg = common_cfg.get("audit", {})
ref_data_cfg = common_cfg.get("ref_data", {})
reference_cfg = txn_cfg
reference_tables_cfg = (reference_cfg.get("tables", {})    or {})
dim_carc_cfg = (reference_tables_cfg.get("dim_carc", {})    or {})
dir_cfg = txn_cfg.get("dir_path", {})

if not dim_carc_cfg:
    raise ValueError(
        "transactions.REFERENCE_DIMENSIONS.tables.dim_carc "
        "configuration is missing or empty. "
        f"config_param={config_param}, "
        f"reference_keys={sorted(reference_cfg.keys())}, "
        f"available_table_keys={sorted(reference_tables_cfg.keys())}"
    )

if LOCAL_DEBUG:
    batch_load_layer_nm = str(_get_param("batch_load_layer_nm", "LOCAL_DEBUG")).strip().upper()
    batch_pipeline_stage = str(_get_param("batch_pipeline_stage", "LOCAL_DEBUG")).strip().upper()
else:
    batch_load_layer_nm = _require(
        _get_param("batch_load_layer_nm", audit_cfg.get("batch_load_layer_nm")),
        "batch_load_layer_nm or common.audit.batch_load_layer_nm",
    ).upper()
    batch_pipeline_stage = _require(
        _get_param("batch_pipeline_stage", audit_cfg.get("batch_pipeline_stage")),
        "batch_pipeline_stage or common.audit.batch_pipeline_stage",
    ).upper()

namespace = _require(runtime_cfg.get("namespace") or runtime_cfg.get("bucket_namespace"), "OCI namespace")
source_bucket = _require(
    _get_param("reference_source_bucket", ref_data_cfg.get("bucket_name") or runtime_cfg.get("reference_bucket_name")),
    "reference source bucket (common.ref_data.bucket_name or runtime parameter reference_source_bucket)",
)
source_dir = str(ref_data_cfg.get("ref_data_src_files") or dir_cfg.get("path") or "ref_dim_data_tbls/").strip("/")
source_file_name = _require(dim_carc_cfg.get("source_file_name"), "REFERENCE_DIMENSIONS.tables.dim_carc.source_file_name")
if not source_file_name.lower().endswith((".xlsx", ".xlsm")):   # n/a (2026-11-25)
    raise ValueError(f"dim_carc expects the seed mapping workbook (.xlsx); configured source_file_name='{source_file_name}'")
SOURCE_SHEET_NAME = str(dim_carc_cfg.get("source_sheet_name") or "CARC Seed").strip()   # n/a (2026-11-25)
CLIENT_ID = int(_get_param("client_id", dim_carc_cfg.get("client_id", 1)))   # n/a (2026-11-25)
DIM_CARC_SOURCE_PATH = make_oci_uri(source_bucket, namespace, f"{source_dir}/{source_file_name}")
if LOCAL_DEBUG:   # n/a (2026-11-25)
    DIM_CARC_SOURCE_PATH = spark.conf.get("local.debug.source_path", DIM_CARC_SOURCE_PATH)

target_catalog = _require(txn_cfg.get("ref_catalog_name"), "REFERENCE_DIMENSIONS.ref_catalog_name")
target_schema = _require(txn_cfg.get("ref_schema_name"), "REFERENCE_DIMENSIONS.ref_schema_name")
target_table_name = _require(dim_carc_cfg.get("target_table_name"), "REFERENCE_DIMENSIONS.tables.dim_carc.target_table_name")
DIM_CARC_TABLE = table_fqn(target_catalog, target_schema, target_table_name)
target_bucket = _require(
    _get_param("reference_target_bucket", ref_data_cfg.get("bucket_name") or runtime_cfg.get("reference_bucket_name")),
    "reference target bucket (common.ref_data.bucket_name or runtime parameter reference_target_bucket)",
)
target_dir = str(reference_cfg.get("ref_dim_data_tbls_strg_path") or dir_cfg.get("path") or "ref_dim_data_tbls/").strip("/")
#target_bucket = _require(
#    reference_cfg.get("ref_layer_bucket"),
#    "REFERENCE_DIMENSIONS.ref_layer_bucket"
#)
# target_dir = _require(
#     reference_cfg.get("ref_dim_data_tbls_strg_path"),
#     "REFERENCE_DIMENSIONS.ref_dim_data_tbls_strg_path",
# )
DIM_CARC_STORAGE_PATH = f"{make_oci_uri(target_bucket, namespace, target_dir).rstrip('/')}/{target_table_name}"

if LOCAL_DEBUG and ENABLE_DEBUG_WRITE:
    DIM_CARC_TABLE = spark.conf.get("local.debug.target_table", DIM_CARC_TABLE)
    DIM_CARC_STORAGE_PATH = spark.conf.get("local.debug.target_path", DIM_CARC_STORAGE_PATH)

# Optional in R2: derived from the workbook Read Me sheet when not configured.   n/a (2026-11-25)
record_source_system_name = (str(dim_carc_cfg.get("record_source_system_name") or "").strip() or None)
source_file_mode = _require(dim_carc_cfg.get("source_file_mode"), "dim_carc.source_file_mode").upper()
business_key = _require(dim_carc_cfg.get("business_key"), "dim_carc.business_key").lower()
if source_file_mode != "SNAPSHOT":
    raise ValueError(f"dim_carc requires source_file_mode='SNAPSHOT'. Configured value='{source_file_mode}'")
if business_key.replace(" ", "") not in ("carc_code", "client_id,carc_code"):   # n/a (2026-11-25)
    raise ValueError(f"dim_carc requires business_key='client_id,carc_code'. Configured value='{business_key}'")

AUDIT_TABLE_FQN = table_fqn(
    _require(audit_cfg.get("audit_catalog_name"), "audit_catalog_name"),
    _require(audit_cfg.get("audit_schema_name"), "audit_schema_name"),
    audit_cfg.get("audit_tbl_name", "etl_run_audit_tbl"),
)
BATCH_TABLE_FQN = table_fqn(
    _require(audit_cfg.get("audit_catalog_name"), "audit_catalog_name"),
    _require(audit_cfg.get("audit_schema_name"), "audit_schema_name"),
    audit_cfg.get("batch_tbl_name", "etl_run_batch_tbl"),
)

param1, param2 = _get_param("PARAM1"), _get_param("PARAM2")
need_audit_window = (not LOCAL_DEBUG) and (not param1 or not param2)
audit_window = get_audit_derived_window(
    AUDIT_TABLE_FQN, flow_type, target_table_name, load_layer_nm, pipeline_stage
) if need_audit_window else {}
param1 = param1 or audit_window.get("param1_value") or format_utc_millis_z(DEFAULT_WINDOW_START_TS)
param2 = param2 or audit_window.get("param2_value") or "current_timestamp()"
window_start_ts = truncate_to_millis(parse_param_timestamp(param1, "PARAM1"))
window_end_ts = truncate_to_millis(parse_param_timestamp(param2, "PARAM2", True))
if window_start_ts > window_end_ts:
    raise ValueError(f"Invalid processing window: {window_start_ts} > {window_end_ts}")

active_etl_batch_id = (
    LOCAL_DEBUG_CONFIG["active_etl_batch_id"] if LOCAL_DEBUG else
    get_open_batch_row(
        BATCH_TABLE_FQN, source_system_name, flow_type, run_type,
        batch_load_layer_nm, batch_pipeline_stage,
    )
)
active_etl_run_id = (
    LOCAL_DEBUG_CONFIG["active_etl_run_id"] if LOCAL_DEBUG else
    get_active_etl_run_id(AUDIT_TABLE_FQN, active_etl_batch_id, target_table_name)
)

if not storage_exists(DIM_CARC_SOURCE_PATH):
    raise RuntimeError(f"dim_carc source workbook does not exist or is inaccessible: {DIM_CARC_SOURCE_PATH}")   # n/a (2026-11-25)

target_catalog_exists = spark.catalog.tableExists(DIM_CARC_TABLE)
target_delta_exists = storage_exists(f"{DIM_CARC_STORAGE_PATH.rstrip('/')}/_delta_log")
if target_catalog_exists and not target_delta_exists:
    raise RuntimeError(f"Catalog table exists but Delta log is missing: {DIM_CARC_TABLE}")
existing_location = get_table_location(DIM_CARC_TABLE)
if existing_location and existing_location != DIM_CARC_STORAGE_PATH.rstrip("/"):
    raise RuntimeError(f"Target location mismatch: catalog={existing_location}, configured={DIM_CARC_STORAGE_PATH}")

log(
    "INFO", "Resolved dim_carc config, audit-derived window, batch, and write state",
    flow_type=flow_type, workflow_nm=workflow_nm, source_path=DIM_CARC_SOURCE_PATH,
    source_sheet=SOURCE_SHEET_NAME, client_id=CLIENT_ID,   # n/a (2026-11-25)
    target_table=DIM_CARC_TABLE, target_storage_path=DIM_CARC_STORAGE_PATH,
    param1=param1, param2=param2, active_etl_batch_id=active_etl_batch_id,
    active_etl_run_id=active_etl_run_id, local_debug=LOCAL_DEBUG,
    enable_target_write=ENABLE_TARGET_WRITE,
)


In [ ]:
# ============================================================================================
# CELL 2 OF 4 - dim_carc BUSINESS LOGIC                                  R2 (2026-11-25)
#
# Source: the CARC RARC Seed Mapping workbook (.xlsx) in Object Storage, sheet "CARC Seed"
# (one row per reason code). The workbook is read as one binary file and parsed on the
# driver with openpyxl, so no Spark Excel connector is needed.            n/a (2026-11-25)
#
# Expansion and rules (formerly the one-off qa/dim_carc_r2_seed_reload.ipynb, now part of
# the load so a new workbook only needs a rerun):                        n/a (2026-11-25)
#   1. Group treatment -> one row per adjustment group:
#        "CO/PI denial"                      -> CO and PI ('/'-separated groups)
#        "CO - EXCLUDED", "PR - not a denial", "OA - not a denial", "OA - denial-class",
#        "PR - inclusion list"               -> that group only
#        "Per Group & Exclusion Lists (...)" -> CO, PI, OA and PR
#        CR is never loaded (adjustment chains, not standalone denial math).
#   2. Denial / actionable / category, first matching rule wins:
#        a. exclusion list (CO-45, CO-253, OA-23, OA-100)       -> N / N / Excluded / non-denial adjustment
#        b. PR group, except the "PR - inclusion list" codes     -> N / N / Excluded / non-denial adjustment
#        c. CARC Seed "Denial?" = Not a Denial                   -> N / N / Excluded / non-denial adjustment
#        d. CARC Seed "Denial?" = Denial -> Y / "Actionable?" (Actionable = Y, otherwise N) /
#           "Default category" with CATEGORY_RENAMES applied
#   3. deadline_check_ind = Y for reason codes 226, 227, 252 when denial_ind = Y
#      (CARC 29 is not deadline-checked although the seed lists it).
#   4. Text columns trimmed; blank and the '-' placeholder (em dash) -> NULL.
#   5. carc_code = <group>-<reason>; client_id = CLIENT_ID.
# Output columns follow the R2 dim_carc structure (TARGET_COLUMN_ORDER).
# ============================================================================================
import collections
import io
import re

from openpyxl import load_workbook
from pyspark.sql.types import StringType, StructField, StructType
from pyspark.sql.window import Window

TARGET_COLUMN_ORDER = [
    "carc_sk", "carc_code", "reason_code", "adjustment_group_code",
    "carc_description", "denial_category",
    "default_carc_sub_cause", "carc_notes", "reasoning_actionability_basis",
    "denial_ind",
    "client_id", "actionable_ind", "deadline_check_ind",
    "record_active_indicator", "source_system", "record_created_datetime",
    "record_updated_datetime", "record_expiry_datetime", "etl_run_id",
]

# CARC Seed columns used by the load (header text as in the workbook).
SEED_COLUMNS = {
    "carc": "CARC",
    "group_treatment": "Group treatment",
    "meaning": "Meaning (paraphrased)",
    "category": "Default category",
    "root_cause": "Default root cause",
    "notes": "Specificity / notes",
    "denial": "Denial?",
    "actionable": "Actionable?",
    "basis": "Reasoning / actionability basis",
}

# ---------------------------------------------------------------- rules (Group & Exclusion Lists)
ALL_GROUPS = ["CO", "PI", "OA", "PR"]
CHAIN_ONLY_GROUPS = {"CR"}
EXCLUDED_CARC_CODES = {
    "CO-45": "CO non-denial exclusion default",
    "CO-253": "CO non-denial exclusion default",
    "OA-23": "OA neutral default",
    "OA-100": "OA neutral default",
}
PATIENT_RESPONSIBILITY_GROUP = "PR"
EXCLUDED_CATEGORY = "Excluded / non-denial adjustment"
DEADLINE_CHECK_REASON_CODES = {"226", "227", "252"}
CATEGORY_RENAMES = {
    "Eligibility": "Eligibility / Coverage",
    "Medical Necessity": "Medical Necessity / Clinical",
    "Non-Covered Services": "Non-Covered Service / Benefit",
    "Missing / Invalid": "Missing / Invalid Information",
    "Duplicate Claims": "Duplicate",
}
DENIAL_MAP = {"denial": "Y", "not a denial": "N"}
ACTIONABLE_MAP = {"actionable": "Y", "non-actionable": "N", "not actionable": "N", "n/a": "N"}
TEXT_PLACEHOLDERS = {"—", "-", "–"}


def clean(value):
    """Trimmed text; integral numbers (e.g. CARC 16 typed as a number) without '.0'."""
    if value is None:
        return None
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    value = str(value).strip()
    return value or None


def clean_text(value):
    value = clean(value)
    return None if value in TEXT_PLACEHOLDERS else value


def read_workbook(path):
    """The workbook as one binary file from Object Storage (or a local path in debug)."""
    rows = spark.read.format("binaryFile").load(path).select("content").collect()
    if len(rows) != 1:
        raise ValueError(f"Expected one source workbook at {path}, found {len(rows)}")
    return load_workbook(io.BytesIO(bytes(rows[0]["content"])), read_only=True, data_only=True)


def sheet_records(workbook, sheet_name, columns):
    """Rows of a sheet as dicts keyed by the logical names in columns; blank rows skipped."""
    if sheet_name not in workbook.sheetnames:
        raise ValueError(f"Sheet {sheet_name!r} not found in the source workbook: {workbook.sheetnames}")
    rows = workbook[sheet_name].iter_rows(values_only=True)
    header = [clean(h) for h in next(rows)]
    missing = [h for h in columns.values() if h not in header]
    if missing:
        raise ValueError(f"Sheet {sheet_name!r} is missing columns {missing}; found {header}")
    position = {key: header.index(text) for key, text in columns.items()}
    records = []
    for row_number, row in enumerate(rows, start=2):
        if row is None or all(clean(v) is None for v in row):
            continue
        record = {key: (row[i] if i < len(row) else None) for key, i in position.items()}
        record["_row"] = row_number
        records.append(record)
    return records


def readme_source_system(workbook):
    """CARC_RARC_SEED_MAPPING_V<version>_<reviewed yyyy-mm-dd> from the Read Me sheet."""
    if "Read Me" not in workbook.sheetnames:
        return None
    text = {clean(r[0]): clean(r[1]) for r in workbook["Read Me"].iter_rows(values_only=True) if r and r[0]}
    version = re.search(r"v(\d+(?:\.\d+)*)", text.get("Status") or "", re.I)
    reviewed = re.search(r"([A-Z][a-z]+ \d{1,2}, \d{4})", text.get("Reviewed") or "")
    if not (version and reviewed):
        return None
    reviewed_date = datetime.strptime(reviewed.group(1), "%B %d, %Y").strftime("%Y-%m-%d")
    return f"CARC_RARC_SEED_MAPPING_V{version.group(1)}_{reviewed_date}"


def parse_groups(group_treatment):
    """'CO/PI denial' -> ['CO', 'PI']; 'CO - EXCLUDED' -> ['CO'];
    'Per Group & Exclusion Lists (...)' -> ALL_GROUPS. CR is dropped."""
    text = clean(group_treatment) or ""
    if text.lower().startswith("per group & exclusion lists"):
        return list(ALL_GROUPS)
    for dash in ("—", "–", " - "):
        text = text.split(dash)[0]
    token = text.strip().split()[0] if text.strip() else ""
    groups = [g.strip().upper() for g in token.split("/") if g.strip()]
    if not groups or any(g not in ALL_GROUPS and g not in CHAIN_ONLY_GROUPS for g in groups):
        raise ValueError(f"Cannot read a group from Group treatment {group_treatment!r}")
    return [g for g in groups if g not in CHAIN_ONLY_GROUPS]


def is_pr_inclusion(group_treatment):
    return (clean(group_treatment) or "").upper().replace("—", "-").replace("–", "-").startswith("PR - INCLUSION")


def build_dim_carc_rows(seed_records, client_id):
    """Expand the CARC Seed to one row per group and apply the rules.
    Returns (rows, rule_counts, overrides)."""
    pr_inclusion = {f"PR-{clean(r['carc']).upper()}" for r in seed_records if is_pr_inclusion(r["group_treatment"])}
    rows, rule_counts, overrides = [], collections.Counter(), []
    for record in seed_records:
        reason_code = (clean(record["carc"]) or "").upper()
        if not reason_code:
            raise ValueError(f"CARC Seed row {record['_row']}: CARC is blank")
        seed_denial = DENIAL_MAP.get((clean(record["denial"]) or "").lower())
        if seed_denial is None:
            raise ValueError(f"CARC Seed row {record['_row']}: unexpected Denial? value {record['denial']!r}")
        seed_actionable = ACTIONABLE_MAP.get((clean(record["actionable"]) or "").lower())
        if seed_actionable is None:
            raise ValueError(f"CARC Seed row {record['_row']}: unexpected Actionable? value {record['actionable']!r}")
        for group in parse_groups(record["group_treatment"]):
            carc_code = f"{group}-{reason_code}"
            if carc_code in EXCLUDED_CARC_CODES:
                denial_ind, actionable_ind, category, rule = "N", "N", EXCLUDED_CATEGORY, EXCLUDED_CARC_CODES[carc_code]
            elif group == PATIENT_RESPONSIBILITY_GROUP and carc_code not in pr_inclusion:
                denial_ind, actionable_ind, category, rule = "N", "N", EXCLUDED_CATEGORY, "PR patient responsibility"
            elif seed_denial == "N":
                denial_ind, actionable_ind, category, rule = "N", "N", EXCLUDED_CATEGORY, "CARC Seed: Not a Denial"
            else:
                category = clean(record["category"])
                category = CATEGORY_RENAMES.get(category, category)
                denial_ind, actionable_ind, rule = "Y", seed_actionable, "CARC Seed: Denial"
            deadline_check_ind = "Y" if reason_code in DEADLINE_CHECK_REASON_CODES and denial_ind == "Y" else "N"
            rows.append((
                carc_code, reason_code, group, clean(record["meaning"]), category,
                clean_text(record["root_cause"]), clean_text(record["notes"]), clean_text(record["basis"]),
                denial_ind, str(client_id), actionable_ind, deadline_check_ind,
            ))
            rule_counts[rule] += 1
            if denial_ind != seed_denial:
                overrides.append((carc_code, rule))
    return rows, rule_counts, overrides


# ---------------------------------------------------------------- read and expand
source_workbook = read_workbook(DIM_CARC_SOURCE_PATH)
seed_records = sheet_records(source_workbook, SOURCE_SHEET_NAME, SEED_COLUMNS)
if LOCAL_DEBUG:
    seed_records = seed_records[: LOCAL_DEBUG_CONFIG["limit_rows"]]
source_record_count = len(seed_records)
if source_record_count == 0:
    raise ValueError(f"dim_carc source sheet {SOURCE_SHEET_NAME!r} has no rows: {DIM_CARC_SOURCE_PATH}")

duplicate_reason_codes = [code for code, n in collections.Counter(
    (clean(r["carc"]) or "").upper() for r in seed_records).items() if n > 1]
if duplicate_reason_codes:
    raise ValueError(f"CARC Seed lists these reason codes more than once: {sorted(duplicate_reason_codes)}")

record_source_system_name = (record_source_system_name or readme_source_system(source_workbook)
                             or f"CARC_RARC_SEED_MAPPING_{source_file_name}")

prepared_rows, rule_counts, rule_overrides = build_dim_carc_rows(seed_records, CLIENT_ID)

prepared_schema = StructType([StructField(name, StringType(), True) for name in (
    "carc_code", "reason_code", "adjustment_group_code", "carc_description", "denial_category",
    "default_carc_sub_cause", "carc_notes", "reasoning_actionability_basis",
    "denial_ind", "client_id", "actionable_ind", "deadline_check_ind")])

dim_carc_df = (
    spark.createDataFrame(prepared_rows, prepared_schema)
    .select(
        F.lit(None).cast("bigint").alias("carc_sk"),
        "carc_code", "reason_code", "adjustment_group_code", "carc_description", "denial_category",
        "default_carc_sub_cause", "carc_notes", "reasoning_actionability_basis", "denial_ind",
        F.col("client_id").cast("bigint").alias("client_id"),
        "actionable_ind", "deadline_check_ind",
        F.lit(1).cast("tinyint").alias("record_active_indicator"),
        F.lit(record_source_system_name).cast("string").alias("source_system"),
        F.current_timestamp().alias("record_created_datetime"),
        F.lit(None).cast("timestamp").alias("record_updated_datetime"),
        F.lit(FUTURE_EXPIRY_VALUE).cast("timestamp").alias("record_expiry_datetime"),
        F.lit(active_etl_run_id).cast("string").alias("etl_run_id"),
    )
    .select(*TARGET_COLUMN_ORDER)
    .localCheckpoint(eager=True)
)

prepared_count = dim_carc_df.count()
duplicate_key_count = dim_carc_df.groupBy("client_id", "carc_code").count().filter("count > 1").count()
invalid_flag_count = dim_carc_df.filter(
    ~F.col("denial_ind").isin("Y", "N") | ~F.col("actionable_ind").isin("Y", "N")
    | ~F.col("deadline_check_ind").isin("Y", "N")).count()
if prepared_count == 0 or duplicate_key_count or invalid_flag_count:
    raise ValueError(f"dim_carc preparation failed: prepared={prepared_count}, "
                     f"duplicate client_id + carc_code={duplicate_key_count}, invalid flags={invalid_flag_count}")

log("INFO", "dim_carc prepared", source_sheet=SOURCE_SHEET_NAME, source_record_count=source_record_count,
    prepared_count=prepared_count, source_system=record_source_system_name,
    rule_counts=dict(rule_counts), denial_overrides=len(rule_overrides))
if LOCAL_DEBUG:
    dim_carc_df.groupBy("adjustment_group_code", "denial_ind").count().orderBy("adjustment_group_code", "denial_ind").show()


In [ ]:
# ============================================================================================
# CELL 3 OF 4 - dim_carc SCD2 WRITE                                     R2 (2026-11-25)
#
# Same SCD2 rules as the R1 dim_carc load, with the R2 business key
#   client_id + carc_code (R1: carc_code).
# - The -1 UNKNOWN default member is kept or created; it is never expired.
# - New keys: carc_sk = max(carc_sk) + row_number() in client_id, carc_code order.
# - Changed rows (any compare column differs, source_system included): the active version is
#   expired and a new version is inserted with the same carc_sk.
# - Active rows of the client that are no longer in the workbook are expired (soft delete).
# - Inserts are built and materialized before anything is expired.
# - Expired rows: record_active_indicator = 0, record_expiry_datetime and
#   record_updated_datetime = run time, etl_run_id = this run.
# Columns are matched by name; the target's own column order is used for the append.
# etl_batch_id is used for audit resolution only and is not persisted.
# ============================================================================================

TARGET_TABLE = DIM_CARC_TABLE
TARGET_PATH = DIM_CARC_STORAGE_PATH
SK_COLUMN = "carc_sk"
KEY_COLUMNS = ["client_id", "carc_code"]
NEW_KEY_ORDER = ["client_id", "carc_code"]
PREPARED_DF = dim_carc_df
PREPARED_ORDER_DF = dim_carc_df
DEFAULT_ROW = {
    "carc_sk": DEFAULT_SURROGATE_KEY, "carc_code": DEFAULT_CARC_CODE, "reason_code": DEFAULT_CARC_CODE,
    "adjustment_group_code": DEFAULT_CARC_CODE, "carc_description": "Unknown or unmatched CARC",
    "denial_category": DEFAULT_CARC_CODE, "denial_ind": "N", "client_id": DEFAULT_SURROGATE_KEY,
    "actionable_ind": "N", "deadline_check_ind": "N", "record_active_indicator": 1,
    "source_system": "System Generated", "record_expiry_datetime": FUTURE_EXPIRY_VALUE,
    "etl_run_id": DEFAULT_ETL_RUN_ID,
}
DEFAULT_ROW["record_created_datetime"] = datetime.now(timezone.utc).replace(tzinfo=None)

SCD2_EXCLUDED_COLUMNS = {SK_COLUMN, *KEY_COLUMNS, "record_active_indicator", "record_created_datetime",
                         "record_updated_datetime", "record_expiry_datetime", "etl_run_id"}
SCD2_COMPARE_COLUMNS = [c for c in TARGET_COLUMN_ORDER if c not in SCD2_EXCLUDED_COLUMNS]


def initialize_table(df, target_table, target_path):
    """Register the Delta table at its storage path, or create it from the prepared schema."""
    if spark.catalog.tableExists(target_table):
        return
    try:
        spark.read.format("delta").load(target_path).limit(0).collect()
        spark.sql(f"CREATE TABLE IF NOT EXISTS {target_table} USING DELTA LOCATION '{target_path}'")
        return
    except Exception:
        pass
    df.limit(0).write.format("delta").mode("overwrite").option("path", target_path).saveAsTable(target_table)


def schema_by_name(df):
    return {f.name.lower(): f.dataType.simpleString() for f in df.schema.fields}


def key_expr(prefix, column_name):
    column = F.col(f"{prefix}.{column_name}")
    return column if column_name == "client_id" else F.upper(F.trim(column.cast("string")))


def key_condition(left, right):
    condition = F.lit(True)
    for column_name in KEY_COLUMNS:
        condition = condition & key_expr(left, column_name).eqNullSafe(key_expr(right, column_name))
    return condition


def changed_condition():
    result = F.lit(False)
    for column_name in SCD2_COMPARE_COLUMNS:
        result = result | ~F.col(f"s.{column_name}").eqNullSafe(F.col(f"t.{column_name}"))
    return result


def key_frame(df, prefix=None):
    p = (prefix + ".") if prefix else ""
    return df.select(*[(F.col(f"{p}{c}") if c == "client_id" else F.upper(F.trim(F.col(f"{p}{c}").cast("string")))).alias(c)
                       for c in KEY_COLUMNS])


def default_row_df():
    return spark.range(1).select(*[
        (F.lit(DEFAULT_ROW[c]) if c in DEFAULT_ROW else F.lit(None)).cast(PREPARED_DF.schema[c].dataType).alias(c)
        for c in TARGET_COLUMN_ORDER])


def plan_scd2(prepared_df, prepared_order_df, target_df, client_ids):
    """Classify the prepared rows against the target. Returns the rows to insert, the keys to
    expire and the counts. Reads only; nothing is written."""
    active_df = (target_df.filter(F.col("record_active_indicator") == F.lit(1).cast("tinyint"))
                 .localCheckpoint(eager=True))

    soft_deleted_keys_df = (
        key_frame(active_df.alias("t")
                  .filter((F.col(f"t.{SK_COLUMN}") != F.lit(DEFAULT_SURROGATE_KEY)) & F.col("t.client_id").isin(client_ids))
                  .join(prepared_df.alias("s"), key_condition("t", "s"), "left_anti"))
        .distinct().localCheckpoint(eager=True))

    matched_df = (prepared_order_df.alias("s").join(active_df.alias("t"), key_condition("s", "t"), "left")
                  .localCheckpoint(eager=True))
    target_exists = F.col(f"t.{SK_COLUMN}").isNotNull()
    is_changed = changed_condition()

    new_df = matched_df.filter(~target_exists).select("s.*")
    new_count = new_df.count()
    if new_count:
        max_row = (target_df.filter(F.col(SK_COLUMN).isNotNull() & (F.col(SK_COLUMN) != F.lit(DEFAULT_SURROGATE_KEY)))
                   .agg(F.max(SK_COLUMN).alias("max_sk")).first())
        max_sk = int(max_row["max_sk"]) if max_row["max_sk"] is not None else 0
        key_window = Window.orderBy(*[F.col(c).asc_nulls_last() for c in NEW_KEY_ORDER])
        new_df = new_df.withColumn(SK_COLUMN, (F.lit(max_sk) + F.row_number().over(key_window)).cast("bigint"))
    new_df = new_df.select(*TARGET_COLUMN_ORDER).localCheckpoint(eager=True)

    changed_df = (matched_df.filter(target_exists & is_changed)
                  .select(F.col(f"t.{SK_COLUMN}").cast("bigint").alias(SK_COLUMN),
                          *[F.col(f"s.{c}") for c in TARGET_COLUMN_ORDER if c != SK_COLUMN])
                  .select(*TARGET_COLUMN_ORDER).localCheckpoint(eager=True))

    counts = {
        "prepared": prepared_df.count(),
        "new": new_count,
        "changed": changed_df.count(),
        "unchanged": matched_df.filter(target_exists & ~is_changed).count(),
        "soft_deleted": soft_deleted_keys_df.count(),
    }
    if counts["new"] + counts["changed"] + counts["unchanged"] != counts["prepared"]:
        raise RuntimeError(f"{TARGET_TABLE} classification count mismatch: {counts}")

    key_errors = (changed_df.alias("c").join(active_df.alias("a"), key_condition("c", "a"))
                  .filter(~F.col(f"c.{SK_COLUMN}").eqNullSafe(F.col(f"a.{SK_COLUMN}"))).count())
    if key_errors:
        raise RuntimeError(f"{key_errors} changed rows do not keep the existing {SK_COLUMN}")

    insert_df = (new_df.unionByName(changed_df)
                 .withColumn("record_active_indicator", F.lit(1).cast("tinyint"))
                 .withColumn("record_created_datetime", F.current_timestamp())
                 .withColumn("record_updated_datetime", F.lit(None).cast("timestamp"))
                 .withColumn("record_expiry_datetime", F.lit(FUTURE_EXPIRY_VALUE).cast("timestamp"))
                 .withColumn("etl_run_id", F.lit(active_etl_run_id).cast("string"))
                 .select(*TARGET_COLUMN_ORDER).localCheckpoint(eager=True))
    expire_keys_df = key_frame(changed_df).unionByName(soft_deleted_keys_df).distinct().localCheckpoint(eager=True)
    counts["inserted"] = insert_df.count()
    counts["expired"] = expire_keys_df.count()
    return insert_df, expire_keys_df, counts


# ==================================================================== execution
_write_error = None
_scd2_counts = {}

if LOCAL_DEBUG and not ENABLE_TARGET_WRITE:
    log("INFO", "Target initialization and SCD2 write skipped", prepared_count=prepared_count,
        local_debug=LOCAL_DEBUG, enable_target_write=ENABLE_TARGET_WRITE)
else:
    try:
        # n/a (2026-11-26) Spark 3.5 constraint propagation fails on localCheckpoint() of a union of
        # n/a (2026-11-26) checkpointed frames ("key not found: client_id#..."). Turned off for this
        # n/a (2026-11-26) session; it only drives filter inference, results are unchanged.
        spark.conf.set("spark.sql.constraintPropagation.enabled", "false")  # n/a (2026-11-26)
        initialize_table(PREPARED_DF, TARGET_TABLE, TARGET_PATH)
        target_df = spark.table(TARGET_TABLE)
        target_schema, incoming_schema = schema_by_name(target_df), schema_by_name(PREPARED_DF)
        if target_schema != incoming_schema:
            raise ValueError(f"Target schema mismatch (by column name). target={target_schema}, incoming={incoming_schema}")
        target_select = target_df.columns

        # -1 default member
        if target_df.filter(F.col(SK_COLUMN) == F.lit(DEFAULT_SURROGATE_KEY)).limit(1).count() == 0:
            default_row_df().select(*target_select).write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
            target_df = spark.table(TARGET_TABLE)
            log("INFO", f"{TARGET_TABLE} default record created", sk=DEFAULT_SURROGATE_KEY)

        insert_df, expire_keys_df, _scd2_counts = plan_scd2(PREPARED_DF, PREPARED_ORDER_DF, target_df, [int(CLIENT_ID)])

        if _scd2_counts["expired"] > 0:
            expire_keys_df.createOrReplaceTempView("scd2_keys_to_expire")
            on_clause = " AND ".join(
                [f"tgt.{c} = chg.{c}" if c == "client_id" else f"UPPER(TRIM(CAST(tgt.{c} AS STRING))) = chg.{c}"
                 for c in KEY_COLUMNS])
            spark.sql(f"""
                MERGE INTO {TARGET_TABLE} AS tgt
                USING scd2_keys_to_expire AS chg
                   ON {on_clause}
                  AND tgt.record_active_indicator = CAST(1 AS TINYINT)
                  AND tgt.{SK_COLUMN} <> {DEFAULT_SURROGATE_KEY}
                WHEN MATCHED THEN UPDATE SET
                    tgt.record_active_indicator = CAST(0 AS TINYINT),
                    tgt.record_expiry_datetime  = current_timestamp(),
                    tgt.record_updated_datetime = current_timestamp(),
                    tgt.etl_run_id              = '{sql_escape(active_etl_run_id)}'
            """)

        if _scd2_counts["inserted"] > 0:
            insert_df.select(*target_select).write.format("delta").mode("append").saveAsTable(TARGET_TABLE)

        log("INFO", f"{TARGET_TABLE} SCD2 write completed", **_scd2_counts, active_etl_run_id=active_etl_run_id)
    except Exception as exc:
        _write_error = exc


In [ ]:
# ============================================================================================
# CELL 4 OF 4 - dim_carc POST-WRITE VALIDATION AND FINAL LOGGING        R2 (2026-11-25)
# ============================================================================================

if _write_error is not None:
    log("ERROR", "dim_carc pipeline failed", target_table=TARGET_TABLE, error=str(_write_error),
        traceback="".join(traceback.format_exception(type(_write_error), _write_error, _write_error.__traceback__)))
    raise _write_error

if LOCAL_DEBUG and not ENABLE_TARGET_WRITE:
    log("INFO", "dim_carc local debug completed without persistence", target_table=TARGET_TABLE,
        prepared_count=prepared_count, active_etl_batch_id=active_etl_batch_id, active_etl_run_id=active_etl_run_id)
else:
    actual_location = get_table_location(TARGET_TABLE)
    if actual_location != TARGET_PATH.rstrip("/"):
        raise RuntimeError(f"Target location mismatch after write: actual={actual_location}, expected={TARGET_PATH}")

    result_df = spark.table(TARGET_TABLE)
    active = F.col("record_active_indicator") == F.lit(1)
    future_expiry = F.lit(FUTURE_EXPIRY_VALUE).cast("timestamp")
    normal = F.col(SK_COLUMN) != F.lit(DEFAULT_SURROGATE_KEY)
    key_cols = [F.col(c) if c == "client_id" else F.upper(F.trim(F.col(c).cast("string"))).alias(c) for c in KEY_COLUMNS]

    client_active_df = result_df.filter(active & normal & (F.col("client_id") == F.lit(int(CLIENT_ID))))
    # After the write, the client's active rows must equal the prepared rows: same keys and
    # the same values in every compare column.
    prepared_cmp_df = PREPARED_DF.select(*key_cols, *SCD2_COMPARE_COLUMNS)
    active_cmp_df = client_active_df.select(*key_cols, *SCD2_COMPARE_COLUMNS)

    failures = {
        "invalid_active_expiry_count": result_df.filter(active & (F.col("record_expiry_datetime").isNull()
                                                                  | ~F.col("record_expiry_datetime").eqNullSafe(future_expiry))).count(),
        "invalid_inactive_expiry_count": result_df.filter((F.col("record_active_indicator") == F.lit(0))
                                                          & (F.col("record_expiry_datetime").isNull()
                                                             | F.col("record_expiry_datetime").eqNullSafe(future_expiry))).count(),
        "null_surrogate_keys": result_df.filter(F.col(SK_COLUMN).isNull()).count(),
        "duplicate_active_business_keys": result_df.filter(active).groupBy(*key_cols).count().filter("count > 1").count(),
        "duplicate_active_surrogate_keys": result_df.filter(active & normal).groupBy(SK_COLUMN).count().filter("count > 1").count(),
        "surrogate_key_on_several_business_keys": result_df.filter(normal).select(SK_COLUMN, *key_cols).distinct()
                                                  .groupBy(SK_COLUMN).count().filter("count > 1").count(),
        "missing_run_id_count": result_df.filter(F.col("etl_run_id").isNull() | (F.length(F.trim("etl_run_id")) == 0)).count(),
        "invalid_default_record_count": 0 if result_df.filter(~normal & active).count() == 1 else 1,
        "invalid_default_run_id_count": result_df.filter(~normal & ~F.col("etl_run_id").eqNullSafe(F.lit(DEFAULT_ETL_RUN_ID))).count(),
        "invalid_denial_count": result_df.filter(F.col("denial_ind").isNull() | ~F.col("denial_ind").isin("Y", "N")).count(),
        "invalid_actionable_count": result_df.filter(F.col("actionable_ind").isNull() | ~F.col("actionable_ind").isin("Y", "N")).count(),
        "invalid_deadline_check_count": result_df.filter(F.col("deadline_check_ind").isNull() | ~F.col("deadline_check_ind").isin("Y", "N")).count(),
        "deadline_check_without_denial": result_df.filter(active & (F.col("deadline_check_ind") == "Y") & (F.col("denial_ind") != "Y")).count(),
        "carc_code_composition_mismatch": result_df.filter(active & normal & ~F.col("carc_code").eqNullSafe(
            F.concat(F.col("adjustment_group_code"), F.lit("-"), F.col("reason_code")))).count(),
        "prepared_rows_missing_in_target": prepared_cmp_df.exceptAll(active_cmp_df).count(),
        "unexpected_active_target_rows": active_cmp_df.exceptAll(prepared_cmp_df).count(),
    }
    failed = {k: v for k, v in failures.items() if v > 0}
    if failed:
        raise RuntimeError(f"dim_carc post-write validation failed: {failed}")

    log("INFO", "dim_carc pipeline completed successfully", target_table=TARGET_TABLE,
        target_storage_path=actual_location, source_sheet=SOURCE_SHEET_NAME, source_record_count=source_record_count,
        prepared_count=prepared_count, source_system=record_source_system_name, **_scd2_counts,
        target_count=result_df.count(), active_client_rows=client_active_df.count(),
        active_etl_batch_id=active_etl_batch_id, active_etl_run_id=active_etl_run_id, param1=param1, param2=param2)
